# Analyse toevoegen handleiding instructies
Dit notebook ondersteunt het verkennen, analyseren en valideren van de handleiding en instructies functionaliteit binnen AskDelphi.  
De module biedt helpermethoden voor het toevoegen, beheren en inspecteren van handleiding en instructie relaties die aan AskDelphi‑topics gekoppeld zijn.  
Het doel van dit notebook is om:

- inzicht te krijgen in de datastructuren en API‑interacties rondom handleidingen en instructies
- voorbeelddata te analyseren en patronen te ontdekken
- helpermethoden te testen en gedrag te evalueren
- experimentele of onderzoeksgerichte analyses uit te voeren ter verbetering van de contentstructuur in AskDelphi

Dit notebook vormt daarmee een flexibel startpunt voor verdere exploratie of debugging van de Handleidingen en instructies zaken.

### Functie toevoegen aan relation.py

In [10]:
def get_relationTypeId_by_relationTypeName(topic_id_action : str, topic_version_id_action: str, relationTypeName: str) -> str:
    # Welke relatie types zijn er?
    endpoint = f"/v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id_action}/topicVersion/{topic_version_id_action}/allowedrelations"
    result = client._request(
        "GET", 
        endpoint,
        json_data={}
    )

    relationTypeId = ""
    for item in result["topicAllowedRelations"]:
        # print(item)
        if item['relationTypeName'] == relationTypeName:
            print(f"{item['relationTypeName']} => relationTypeId {item["relationTypeId"]}")
            relationTypeId = item["relationTypeId"]
            break
    
    return relationTypeId

### Connectie met AskDelphi opzetten

In [11]:
import uuid
from ask_delphi_api.authentication import AskDelphiClient
client = AskDelphiClient()
client.authenticate()   # pakt automatisch portal code uit .env

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


True

In [12]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.relation import Relation
from ask_delphi_api.workflow import Workflow
from datetime import datetime
workflow = Workflow(client)
project = Project(client)
topic = TopicTools(client, project)
relation = Relation(client)

### Uitvragen RelationTypeId Handleidingen en instructies

In [13]:
# Checkout Taak topic
topic_id = "3a1f20cc-47d1-461a-be9e-75c603ec7e2f" #Taak Beschikking Uitbrengen ba
result = topic.checkout(topic_id)
topic_version_id = result['topicVersionId']
print(f"topic_id : {topic_id}")
print(f"topic_version_id : {topic_version_id}")

# RelationTypeId Handleidingen en instructies uitvragen
parentTopicId = topic_id
parentTopicVersionId = topic_version_id
parentTopicRelationTypeId = get_relationTypeId_by_relationTypeName(topic_id, topic_version_id, "Handleidingen en instructies")
print(f"parentTopicRelationTypeId : {parentTopicRelationTypeId}")

# Welke relatie types zijn er?
# endpoint = f"/v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id}/topicVersion/{topic_version_id}/allowedrelations"
# result = client._request(
#     "GET", 
#     endpoint,
#     json_data={}
# )

# relationTypeId verschilt per topicType
# for item in result["topicAllowedRelations"]:
#     if item['pyramidLevelName'] == "Handleidingen en Instructies":
#         # print(item['topicTypeName'], item["relationTypeId"])
#         print(item)

topic_id : 3a1f20cc-47d1-461a-be9e-75c603ec7e2f
topic_version_id : 5cf10295-a2cf-4cf6-8852-a939ae2be5dc
Handleidingen en instructies => relationTypeId d03ef735-d495-4f14-8509-94e39474da71
parentTopicRelationTypeId : d03ef735-d495-4f14-8509-94e39474da71


### Aanmaken Externe link

In [14]:
# Creatie topic ID Externe URL
topic_id_externe_url = str(uuid.uuid4())      
topicTitle = "Externe URL download 3"      
topicTypeId = project.get_topic_type_id("External URL")

# Toevoegen Externe URL topic aan Taak topic
relation.add_topic_with_relation(client, topic_id_externe_url, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)

# Checkout Externe URL topic.
result = topic.checkout(topic_id_externe_url)
topic_version_id_externe_url = result['topicVersionId']

# Haal topic-parts op.
content = topic.get_topic_parts(topicId=topic_id_externe_url)

# Selecteer part uit topic met daarin de content.
body_part = None
groups = content['topicEditorData']['groups']
for group in groups:
    for part in group['parts']:
        if part["defaultLabel"] == "Link metadata":
            body_part = part

# Pas content topic aan.
current_date = datetime.now()
topic.topic_add_link(topicVersionId=topic_version_id_externe_url, topicId=topic_id_externe_url, partId="link-meta-data", part=body_part, new_text='https://www.ariebaas.nl')


{}

In [15]:
# Checkin Externe URL topic
topic.checkin(topic_id_externe_url)
print(f"Checked in Externe URL topic : {topic_id_externe_url}")

Checked in Externe URL topic : dfdb5000-ed5c-4b2c-8606-dfd251a718e6


In [16]:
# Checkin Taak topic
topic.checkin(topic_id)
print(f"Checked in task topic : {topic_id}")

Checked in task topic : 3a1f20cc-47d1-461a-be9e-75c603ec7e2f
